# Liquidity Predictor
Welcome to the eighth lesson! This Jupyter Notebook file is meant to accompany **L08 - Liquidity Predictor.**

Type your solutions for each exercise in the code cells below, and then press **Shift + Enter** to execute your code. Then, check the solution video to see how you did!

### 1. Liquidity Predictor Introduction

In [31]:
import pandas as pd
df_client = pd.read_csv('../data/raw/liquidity_client.csv')

In [32]:
df_client.head(3)

,sp_score,market_cap,total_debt,ltm_capex,ltm_ebitda,ltm_fcf,ltm_revenue
0,2,54856.1961,84628,-9262,21387.00032,9488,170315


### 2. Data Exploration

![Exercise 2.1](../figures/exercise_2_1.png)

<font color = 'blue'> **EXERCISE 2.1** </font>

In [68]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

In [34]:
df_liquidity = pd.read_csv('../data/raw/liquidity_data.csv')
df_liquidity.head(3)

,available_liquidity,sp_score,market_cap,total_debt,ltm_capex,ltm_ebitda,ltm_fcf,ltm_revenue
0,28694.04271,2,54856.1961,84628.0,-9262.0,21387.00032,9488.0,170315.0
1,24784.00051,7,209150.6401,57909.0,-2021.0,15161.00019,12105.0,37727.0
2,24142.00013,6,180108.3453,32970.0,-1817.0,15818.99981,12604.0,192592.0


In [35]:
df_liquidity.shape

(802, 8)

<font color = 'blue'> **EXERCISE 2.2** </font>

![Exercise 2.2](../figures/exercise_2_2.png)

![liquidity metadata descriptions](../figures/liquidity_metadata.png)

In [36]:
df_liquidity.describe()

,available_liquidity,sp_score,market_cap,total_debt,ltm_capex,ltm_ebitda,ltm_fcf,ltm_revenue
count,802.000000,802.000000,802.000000,802.000000,802.00000,802.000000,802.000000,802.000000
mean,3884.952199,3.017456,41645.089870,9040.720589,-1200.91799,3455.752891,1772.335973,20420.383638
std,4267.893247,1.851461,74046.522440,12112.161513,2066.29159,5679.466199,4207.345101,39483.422972
min,267.000000,0.000000,4282.810112,0.000000,-15858.00000,-6530.000000,-4888.000000,503.586000
25%,1288.328992,2.000000,10082.091010,2125.297000,-1241.00000,936.200000,232.923000,3791.525000
50%,2395.700000,3.000000,19349.403650,4562.000000,-512.00000,1695.700032,682.110000,8587.166000
75%,4512.458596,4.000000,41154.826240,10478.000000,-190.00000,3706.000000,1788.000000,18816.000000
max,28694.042710,10.000000,777070.706700,87032.000000,-5.04900,69715.000320,53244.000000,487511.000000


### 3. Splitting Data

**Lesson Workspace**

In [37]:
# Import train_test_split() from Scikit Learn's model_selection module

#  This was performed at the beginning of the notebook.  I like to keep all the imports at the top.


<font color = 'blue'> **EXERCISE 3.1** </font>

![Exercise 3.1](../figures/exercise_3_1.png)

![drop.png: explanation of using the drop command](../figures/drop.png)

In [38]:
# Define a Series named "target" containing only the target variable
target = df_liquidity.available_liquidity

# Define a DataFrame named "inputs" containing only the input features
inputs = df_liquidity.drop(columns=['available_liquidity'])


In [39]:
# Display the first row of target
target.head(1)


0    28694.04271
Name: available_liquidity, dtype: float64

In [40]:
# Display the first row of inputs
inputs.head(1)


,sp_score,market_cap,total_debt,ltm_capex,ltm_ebitda,ltm_fcf,ltm_revenue
0,2,54856.1961,84628.0,-9262.0,21387.00032,9488.0,170315.0


**Lesson Workspace**

In [41]:
# Split your data and pass the results to a new object named "results"
results = train_test_split(inputs, target, test_size=0.2, random_state=1)


In [42]:
# Print the type() and len() of results
print(type(results))
print(len(results))
print('---')

# For each item in results, print the item's dimensions
for item in results:
    print(item.shape)


<class 'list'>
4
---
(641, 7)
(161, 7)
(641,)
(161,)


<font color = 'blue'> **EXERCISE 3.2** </font>

![exercise 3.2](../figures/exercise_3_2.png)

In [43]:
# Assign the results to the following variables: input_train, input_test, target_train, target_test

input_train, input_test, target_train, target_test = results

**Lesson Workspace**

In [44]:
input_train.head(1)

,sp_score,market_cap,total_debt,ltm_capex,ltm_ebitda,ltm_fcf,ltm_revenue
309,3,16764.94643,1887.019,-483.002,1905.83296,859.71,8653.205


In [45]:
input_test.head(1)

,sp_score,market_cap,total_debt,ltm_capex,ltm_ebitda,ltm_fcf,ltm_revenue
8,8,172479.4798,24842.0,-1674.0,10841.00006,6830.0,39929.0


In [46]:
target_train.head(1)

309    1227.539
Name: available_liquidity, dtype: float64

In [47]:
target_test.head(1)

8    17708.00026
Name: available_liquidity, dtype: float64

### 4. Model Pipelines

![Model Pipelines](../figures/model_pipelines.png)


**Lesson Workspace**

In [48]:
# Import required functions from Scikit-Learn
# The libraries were imported at the beginning of the notebook.

# Create pipelines dictionary with model pipelines for Lasso and Ridge
pipelines = {
    'lasso' : make_pipeline(StandardScaler(), Lasso(random_state=1)),
    'ridge' : make_pipeline(StandardScaler(), Ridge(random_state=1)),
}


In [49]:
# Add pipeline for Elastic Net
# Dict_name['key'] = value

pipelines['enet'] = make_pipeline(StandardScaler(), ElasticNet(random_state=1))


In [50]:
pipelines

{'lasso': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('lasso', Lasso(random_state=1))]),
 'ridge': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('ridge', Ridge(random_state=1))]),
 'enet': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('elasticnet', ElasticNet(random_state=1))])}

<font color = 'blue'> **EXERCISE 4.1** </font>

![Exercise 4.1](../figures/exercise_4_1.png)

In [58]:
# Dict_name['key'] = value
pipelines['rf'] = make_pipeline(StandardScaler(), RandomForestRegressor(random_state=1))
pipelines['gb'] = make_pipeline(StandardScaler(), GradientBoostingRegressor(random_state=1))

In [59]:
pipelines

{'lasso': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('lasso', Lasso(random_state=1))]),
 'ridge': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('ridge', Ridge(random_state=1))]),
 'enet': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('elasticnet', ElasticNet(random_state=1))]),
 'rf': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('randomforestregressor',
                  RandomForestRegressor(random_state=1))]),
 'gb': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('gradientboostingregressor',
                  GradientBoostingRegressor(random_state=1))])}

In [60]:
for key, value in pipelines.items():
    print(key, type(value))

lasso <class 'sklearn.pipeline.Pipeline'>
ridge <class 'sklearn.pipeline.Pipeline'>
enet <class 'sklearn.pipeline.Pipeline'>
rf <class 'sklearn.pipeline.Pipeline'>
gb <class 'sklearn.pipeline.Pipeline'>


**Lesson Workspace**

In [51]:
# Run this cell to verify that all 5 pipelines are properly defined
for key, value in pipelines.items():
    print(key, type(value))

lasso <class 'sklearn.pipeline.Pipeline'>
ridge <class 'sklearn.pipeline.Pipeline'>
enet <class 'sklearn.pipeline.Pipeline'>


### 5. Hyperparameter Tuning

**Lesson Workspace**

In [64]:
# Create a hyperparameter grid for Lasso

"""
The key must follow a specific order.  Must start with the pipeline, double '_', then must be the hyperparameter.  ie: lasso__alpha

alpha is = 0.1, so values above and below it were added to the list to see what works the best.
"""

lasso_hyperparameters = {
    'lasso__alpha' : [0.01, 0.05, 0.1, 0.5, 1, 5],
}


<font color = 'blue'> **EXERCISE 5.1** </font>

![Exercise 5.1](../figures/exercise_5_1.png)

In [65]:
# Create a hyperparameter grid for Ridge
ridge_hyperparameters = {
    'ridge__alpha' : [0.01, 0.05, 0.1, 0.5, 1, 5]
}


# Create a hyperparameter grid for Elastic Net
elasticnet_hyperparameters = {
    'elasticnet__alpha' : [0.01, 0.05, 0.1, 0.5, 1, 5],
    'elasticnet__l1_ratio' : [0.1, 0.3, 0.5, 0.7, 0.9],
}


<font color = 'blue'> **EXERCISE 5.2** </font>

![Exercise 5.2](../figures/exercise_5_2.png)

![Exercise 5.2b](../figures/exercise_5_2b.png)

In [62]:
# Create a hyperparameter grid for Random Forest
rf_hyperparameters = {
    'randomforestregressor__n_estimators' : [100, 200],
    'randomforestregressor__max_features' : ['auto', 0.3, 0.6],
}

# Create a hyperparameter grid for Gradient Booster
gb_hyperparameters= {
    'gradientboostingregressor__n_estimators' : [100, 200],
    'gradientboostingregressor__learning_rate' : [0.05, 0.1, 0.2],
    'gradientboostingregressor__max_depth' : [1, 3, 5],
}


**Lesson Workspace**

In [66]:
# Create the hyperparameter_grids dictionary

hyperparameter_grids = {
    'lasso' : lasso_hyperparameters,
    'ridge' : ridge_hyperparameters,
    'enet' : elasticnet_hyperparameters,
    'rf': rf_hyperparameters,
    'gb' : gb_hyperparameters,
}

In [67]:
# Run the code below to make sure everything is set up correctly
for key in ['enet', 'gb', 'ridge', 'rf', 'lasso']:
    if key in hyperparameter_grids:
        if type(hyperparameter_grids[key]) is dict:
            print( key, 'was found, and it is a grid.' )
        else:
            print( key, 'was found, but it is not a grid.' )
    else:
        print( key, 'was not found')

enet was found, and it is a grid.
gb was found, and it is a grid.
ridge was found, and it is a grid.
rf was found, and it is a grid.
lasso was found, and it is a grid.


### 6. Cross Validation

**Lesson Workspace**

In [69]:
# Import GridSearchCV
untrained_lasso_model = GridSearchCV(pipelines['lasso'], hyperparameter_grids['lasso'], cv=5)


In [71]:
untrained_lasso_model

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('lasso', Lasso(random_state=1))]),
             param_grid={'lasso__alpha': [0.01, 0.05, 0.1, 0.5, 1, 5]})

In [70]:
print(pipelines.keys())
print('---')
print(hyperparameter_grids.keys())

dict_keys(['lasso', 'ridge', 'enet', 'rf', 'gb'])
---
dict_keys(['lasso', 'ridge', 'enet', 'rf', 'gb'])


<font color = 'blue'> **EXERCISE 6.1** </font>

![Exercise 6.1](../figures/exercise_6_1.png)

In [74]:
models ={}
for key in pipelines.keys():
    models[key] = GridSearchCV(pipelines[key], hyperparameter_grids[key], cv=5)
print(models.keys())

dict_keys(['lasso', 'ridge', 'enet', 'rf', 'gb'])


In [75]:
models['lasso'].fit(input_train, target_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('lasso', Lasso(random_state=1))]),
             param_grid={'lasso__alpha': [0.01, 0.05, 0.1, 0.5, 1, 5]})

<font color = 'blue'> **EXERCISE 6.2** </font>

### 7. Selecting a Winning Model

**Lesson Workspace**

In [ ]:
# Import the r-squared and mean absolute error metrics



<font color = 'blue'> **EXERCISE 7.1** </font>

<font color = 'blue'> **EXERCISE 7.2** </font>

In [ ]:
# Make prediction with test data


# Plot predictions on x axis and actuals on y axis


# Label axes and show graph


